In [4]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# EMB_PATH      = Path("/home/adickson/rice_data/sativas413_embeddings.pt")
# PHENO_PATH    = Path("../rice_data/RiceDiversity_44K_Phenotypes_34traits_PLINK.txt")
EMB_PATH      = Path("/home/andrew.dickson/svar/embeddings/sativas413_embeddings.pt")
PHENO_PATH    = Path("/home/andrew.dickson/rice_data/RiceDiversity_44K_Phenotypes_34traits_PLINK.txt")

data = torch.load(EMB_PATH, map_location="cpu", weights_only=True)
embeddings  = data["embeddings"].float().numpy()   # (413, 768)
emb_samples = data["sample_ids"]

# Center embeddings 
embeddings_centered = embeddings - embeddings.mean(axis=0)

print(f"Shape (samples × dims) : {embeddings.shape}")
print(f"First 5 sample IDs: {emb_samples[:5]}")

from sklearn.preprocessing import StandardScaler
embeddings_scaled = StandardScaler().fit_transform(embeddings_centered)

pheno_raw = pd.read_csv(PHENO_PATH, sep="\t")

TRAIT_COLS = [c for c in pheno_raw.columns if c not in ("HybID", "NSFTVID")]
print(f"{len(TRAIT_COLS)} trait columns.")
EVAL_TRAITS = ["Alkali spreading value",
"Amylose content",
"Panicle number per plant",
"Protein content",
"Seed length",
"Seed number per panicle"]

# Extract NSFTVID from embedding sample names (suffix after last '_')
emb_nsftvid = [int(s.rsplit("_", 1)[-1]) for s in emb_samples]
emb_id_to_idx = {nid: i for i, nid in enumerate(emb_nsftvid)}

# Keep only phenotype rows that have a matching valid embedding sample
pheno_raw["NSFTVID"] = pheno_raw["NSFTVID"].astype(int)
pheno_matched = pheno_raw[pheno_raw["NSFTVID"].isin(emb_id_to_idx)].copy()
pheno_matched = pheno_matched.reset_index(drop=True)

print(f"Phenotype rows with matching embedding: {len(pheno_matched)} / {len(pheno_raw)}")

# Reorder embeddings rows to match phenotype order
emb_row_order = [emb_id_to_idx[nid] for nid in pheno_matched["NSFTVID"]]
X = embeddings_scaled[emb_row_order, :]   # reordered matrix

print(f"Aligned X shape: {X.shape}")

# Build the scaled phenotype array
Y_raw = pheno_matched[TRAIT_COLS].values.astype(float)   # (n_samples, n_traits)
Y_scaled = np.full_like(Y_raw, np.nan)
for j in range(Y_raw.shape[1]):
    col = Y_raw[:, j]
    mask = ~np.isnan(col)
    if mask.sum() < 2:
        continue
    mu  = col[mask].mean()
    std = col[mask].std(ddof=0)
    Y_scaled[mask, j] = (col[mask] - mu) / (std if std > 0 else 1.0)

print(f"Y_scaled shape : {Y_scaled.shape}  (samples × traits)")


Shape (samples × dims) : (383, 768)
First 5 sample IDs: ['081215-A05_1', '081215-A06_3', '081215-A07_4', '081215-A08_5', '090414-A09_6']
36 trait columns.
Phenotype rows with matching embedding: 383 / 413
Aligned X shape: (383, 768)
Y_scaled shape : (383, 36)  (samples × traits)


In [5]:
import math
from scipy.stats import pearsonr
from sklearn.metrics import make_scorer, mean_absolute_error
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.cross_decomposition import PLSRegression


def pearson_correlation_scorer(y_true, y_pred):
    if len(y_true) < 2:
        return 0.0
    r, _ = pearsonr(y_true, y_pred)
    return r if not math.isnan(r) else 0.0


pcc_scorer = make_scorer(pearson_correlation_scorer, greater_is_better=True)
mae_scorer = make_scorer(mean_absolute_error,        greater_is_better=False)

scoring = {
    "MAE": make_scorer(mean_absolute_error, greater_is_better=False),
    "PCC": pcc_scorer,
}

# ── Param grids ───────────────────────────────────────────────────────────────

params_rr = {
    "pca__n_components": [50, 100, 200],
    "model__alpha": np.logspace(-4, 4, 9).tolist(),
}

params_svr = {
    "pca__n_components": [50, 100, 200],
    "model__C":      np.logspace(-2, 3, 6).tolist(),
    "model__gamma":  np.logspace(-4, 1, 6).tolist(),
    "model__kernel": ["rbf", "poly"],
}

params_rf = {
    "pca__n_components":    [50, 100, 200],
    "model__n_estimators":  [100, 200, 500],
    "model__max_depth":     [3, 5, 10, None],
    "model__min_samples_leaf": [1, 2, 4],
}

params_gbr = {
    "pca__n_components":       [50, 100, 200],
    "model__n_estimators":     [100, 200, 500],
    "model__learning_rate":    [0.01, 0.05, 0.1],
    "model__max_depth":        [3, 5],
    "model__min_samples_leaf": [1, 2, 4],
}

params_pls = {
    "n_components": list(range(2, 21)),
}

# ── Pipelines ─────────────────────────────────────────────────────────────────

n_splits    = 5
cv_strategy = KFold(n_splits=n_splits, shuffle=True, random_state=42)

pipe_rr  = Pipeline([("pca", PCA()), ("model", Ridge())])
pipe_svr = Pipeline([("pca", PCA()), ("model", SVR())])
pipe_rf  = Pipeline([("pca", PCA()), ("model", RandomForestRegressor(random_state=42))])
pipe_gbr = Pipeline([("pca", PCA()), ("model", GradientBoostingRegressor(random_state=42))])

sklearn_pipelines = [
    ("RR-BLUP/Ridge",     pipe_rr,  params_rr),
    ("SVR",               pipe_svr, params_svr),
    # ("Random Forest",     pipe_rf,  params_rf),
    # ("Gradient Boosting", pipe_gbr, params_gbr),
]

print("Imports and config ready.")
print(f"  {len(EVAL_TRAITS)} traits, {n_splits}-fold CV, {len(sklearn_pipelines)} sklearn models + PLS")


Imports and config ready.
  6 traits, 5-fold CV, 2 sklearn models + PLS


In [6]:
results = []

for j, trait in enumerate(EVAL_TRAITS):
    y_col = Y_scaled[:, j]
    mask  = ~np.isnan(y_col)
    if mask.sum() < n_splits * 2:
        print(f"[{j+1}/{len(EVAL_TRAITS)}] {trait}: skipped (only {mask.sum()} non-NaN samples)")
        continue

    X_t = X[mask]
    y_t = y_col[mask]

    print(f"\n{'='*60}")
    print(f"[{j+1}/{len(EVAL_TRAITS)}] Trait: {trait}  (n={mask.sum()})")

    best_pcc    = -np.inf
    best_mae    = np.inf
    best_params = None
    best_model  = None

    for name, pipeline, param_grid in sklearn_pipelines:
        gs = GridSearchCV(
            estimator  = pipeline,
            param_grid = param_grid,
            scoring    = scoring,
            refit      = "PCC",
            cv         = cv_strategy,
            n_jobs     = -1,
            verbose    = 0,
        )
        gs.fit(X_t, y_t)

        idx      = gs.best_index_
        pcc_     = gs.cv_results_["mean_test_PCC"][idx]
        mae_     = -gs.cv_results_["mean_test_MAE"][idx]
        params_  = gs.best_params_

        print(f"  {name:<22}  PCC={pcc_:.4f}  MAE={mae_:.4f}  {params_}")

        if pcc_ > best_pcc:
            best_pcc, best_mae, best_params, best_model = pcc_, mae_, params_, name

    # PLS (no PCA/scaler prefix)
    gs_pls = GridSearchCV(
        estimator  = PLSRegression(),
        param_grid = params_pls,
        scoring    = scoring,
        refit      = "PCC",
        cv         = cv_strategy,
        n_jobs     = -1,
        verbose    = 0,
    )
    gs_pls.fit(X_t, y_t)

    idx_pls  = gs_pls.best_index_
    pcc_pls  = gs_pls.cv_results_["mean_test_PCC"][idx_pls]
    mae_pls  = -gs_pls.cv_results_["mean_test_MAE"][idx_pls]

    print(f"  {'PLS':<22}  PCC={pcc_pls:.4f}  MAE={mae_pls:.4f}  {gs_pls.best_params_}")

    if pcc_pls > best_pcc:
        best_pcc, best_mae, best_params, best_model = pcc_pls, mae_pls, gs_pls.best_params_, "PLS"

    print(f"  >> Best: {best_model}  PCC={best_pcc:.4f}  MAE={best_mae:.4f}")

    results.append({
        "Trait":      trait,
        "Model":      best_model,
        "PCC":        round(best_pcc,  4),
        "MAE":        round(best_mae,  4),
        "BestParams": best_params,
        "n_samples":  int(mask.sum()),
    })

results_df = pd.DataFrame(results).sort_values("PCC", ascending=False).reset_index(drop=True)
print("\n\n" + "="*60)
print("FINAL RESULTS (sorted by PCC)")
print(results_df[["Trait", "Model", "PCC", "MAE", "n_samples"]].to_string(index=False))



[1/6] Trait: Alkali spreading value  (n=347)
  RR-BLUP/Ridge           PCC=0.5159  MAE=0.6625  {'model__alpha': 1000.0, 'pca__n_components': 100}


/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: ConstantInputWarning:

  SVR                     PCC=0.5882  MAE=0.6148  {'model__C': 10.0, 'model__gamma': 0.001, 'model__kernel': 'rbf', 'pca__n_components': 200}
  PLS                     PCC=0.4714  MAE=0.6883  {'n_components': 3}
  >> Best: SVR  PCC=0.5882  MAE=0.6148

[2/6] Trait: Amylose content  (n=284)
  RR-BLUP/Ridge           PCC=0.4580  MAE=0.6553  {'model__alpha': 100.0, 'pca__n_components': 200}


/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarning: An input array is nearly constant; the co

  SVR                     PCC=0.4655  MAE=0.6420  {'model__C': 10.0, 'model__gamma': 0.001, 'model__kernel': 'rbf', 'pca__n_components': 200}
  PLS                     PCC=0.4321  MAE=0.6809  {'n_components': 6}
  >> Best: SVR  PCC=0.4655  MAE=0.6420

[3/6] Trait: Panicle number per plant  (n=334)
  RR-BLUP/Ridge           PCC=0.3953  MAE=0.6609  {'model__alpha': 100.0, 'pca__n_components': 200}


...

  SVR                     PCC=0.4459  MAE=0.6535  {'model__C': 1.0, 'model__gamma': 0.001, 'model__kernel': 'rbf', 'pca__n_components': 200}
  PLS                     PCC=0.3114  MAE=0.6949  {'n_components': 4}
  >> Best: SVR  PCC=0.4459  MAE=0.6535

[5/6] Trait: Seed length  (n=283)
  RR-BLUP/Ridge           PCC=0.4184  MAE=0.6921  {'model__alpha': 100.0, 'pca__n_components': 100}


/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarn

  SVR                     PCC=0.4719  MAE=0.6445  {'model__C': 10.0, 'model__gamma': 0.001, 'model__kernel': 'rbf', 'pca__n_components': 100}
  PLS                     PCC=0.4035  MAE=0.6721  {'n_components': 3}
  >> Best: SVR  PCC=0.4719  MAE=0.6445

[6/6] Trait: Seed number per panicle  (n=356)
  RR-BLUP/Ridge           PCC=0.5441  MAE=0.6673  {'model__alpha': 100.0, 'pca__n_components': 50}


/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, _ = pearsonr(y_true, y_pred)
/local/scratch/andrew.dickson/19715147/ipykernel_2488667/3886748655.py:16: ConstantInputWarning:

  SVR                     PCC=0.6317  MAE=0.6099  {'model__C': 10.0, 'model__gamma': 0.01, 'model__kernel': 'rbf', 'pca__n_components': 200}
  PLS                     PCC=0.5577  MAE=0.6627  {'n_components': 5}
  >> Best: SVR  PCC=0.6317  MAE=0.6099


FINAL RESULTS (sorted by PCC)
                   Trait Model    PCC    MAE  n_samples
 Seed number per panicle   SVR 0.6317 0.6099        356
  Alkali spreading value   SVR 0.5882 0.6148        347
             Seed length   SVR 0.4719 0.6445        283
         Amylose content   SVR 0.4655 0.6420        284
         Protein content   SVR 0.4459 0.6535        325
Panicle number per plant   SVR 0.4449 0.5872        334


In [7]:
print("\n\n" + "="*60)
print("FINAL RESULTS (sorted by PCC)")
print(results_df[["Trait", "Model", "PCC", "MAE", "n_samples"]].to_string(index=False))



FINAL RESULTS (sorted by PCC)
                   Trait Model    PCC    MAE  n_samples
 Seed number per panicle   SVR 0.6317 0.6099        356
  Alkali spreading value   SVR 0.5882 0.6148        347
             Seed length   SVR 0.4719 0.6445        283
         Amylose content   SVR 0.4655 0.6420        284
         Protein content   SVR 0.4459 0.6535        325
Panicle number per plant   SVR 0.4449 0.5872        334




FINAL RESULTS (sorted by PCC)
                   Trait Model    PCC    MAE  n_samples
 Seed number per panicle   SVR 0.6944 0.5486        384
  Alkali spreading value   SVR 0.5883 0.5956        374
Panicle number per plant   SVR 0.5372 0.5921        359
         Protein content   SVR 0.5056 0.6176        349
             Seed length   SVR 0.4931 0.6534        304
         Amylose content   SVR 0.4314 0.6606        305


In [22]:
============================================================
FINAL RESULTS (sorted by PCC)
                   Trait Model    PCC    MAE  n_samples
 Seed number per panicle   SVR 0.6906 0.5698        384
  Alkali spreading value   SVR 0.6182 0.5778        374
         Protein content   SVR 0.5440 0.6053        349
Panicle number per plant   SVR 0.4948 0.5909        359
         Amylose content   SVR 0.4624 0.6445        305
             Seed length   SVR 0.4598 0.6707        304

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 4)

In [ ]:
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
results_df[["Trait", "Model", "PCC", "MAE", "n_samples"]].to_csv(
    RESULTS_DIR / "snp_emb_grid_search_results.csv", index=False
)
print("Saved emb_grid_search_results.csv")

Saved emb_grid_search_results.csv
